In [2]:
import sys; sys.path.append("..")
import numpy as np, torch, torch.nn as nn
from data import get_splits, one_hot
from mlp import MLP, train_mlp, eval_loss_acc

d = get_splits()
Xtr, ytr, Xva, yva = d["X_train"], d["y_train"], d["X_val"], d["y_val"]

def to_torch(net, dtype=torch.float64):
    """Sao chép tham số của MLP numpy sang mạng PyTorch tương đương (chuyển vị W)."""
    H = net.W1.shape[1]
    m = nn.Sequential(nn.Linear(784, H), nn.ReLU(), nn.Linear(H, 10)).double()
    with torch.no_grad():
        m[0].weight.copy_(torch.from_numpy(net.W1.T.astype(np.float64).copy()))
        m[0].bias.copy_(torch.from_numpy(net.b1.astype(np.float64).copy()))
        m[2].weight.copy_(torch.from_numpy(net.W2.T.astype(np.float64).copy()))
        m[2].bias.copy_(torch.from_numpy(net.b2.astype(np.float64).copy()))
    return m.to(dtype)

def rel(a, b):
    """Sai số lớn nhất, chia cho độ lớn lớn nhất của a (tránh chia cho phần tử gần 0)."""
    return np.abs(a - b).max() / max(np.abs(a).max(), 1e-12)

In [3]:
net = MLP(n_hidden=128, seed=0, dtype=np.float64)
X = Xtr[:256].astype(np.float64); y = ytr[:256]; Y = one_hot(y).astype(np.float64)

loss_np, g = net.loss_and_grads(X, Y)

m = to_torch(net)
loss_t = nn.CrossEntropyLoss()(m(torch.from_numpy(X)), torch.from_numpy(y))
loss_t.backward()

print("loss numpy :", loss_np)
print("loss torch :", loss_t.item())
print("|chênh lệch loss|:", abs(loss_np - loss_t.item()))
for name, gt, gn in [("W1", m[0].weight.grad.numpy().T, g["W1"]),
                     ("b1", m[0].bias.grad.numpy(),     g["b1"]),
                     ("W2", m[2].weight.grad.numpy().T, g["W2"]),
                     ("b2", m[2].bias.grad.numpy(),     g["b2"])]:
    print(f"{name}: sai số lớn nhất {np.abs(gn - gt).max():.2e} | tương đối {rel(gn, gt):.2e}")

loss numpy : 2.438712367015345
loss torch : 2.438712367015345
|chênh lệch loss|: 0.0
W1: sai số lớn nhất 2.78e-17 | tương đối 5.83e-16
b1: sai số lớn nhất 2.08e-17 | tương đối 6.22e-16
W2: sai số lớn nhất 2.78e-17 | tương đối 3.07e-16
b2: sai số lớn nhất 2.78e-17 | tương đối 2.98e-16


In [4]:
rng = np.random.default_rng(123)
net = MLP(n_hidden=128, seed=0, dtype=np.float64)
m = to_torch(net)
opt = torch.optim.SGD(m.parameters(), lr=0.1)
ce = nn.CrossEntropyLoss()

for step in range(200):
    idx = rng.choice(len(Xtr), 128, replace=False)
    xb = Xtr[idx].astype(np.float64); yb = ytr[idx]
    _, gr = net.loss_and_grads(xb, one_hot(yb).astype(np.float64))
    net.apply(gr, 0.1)                                   # bước của bạn
    opt.zero_grad()                                      # bước của PyTorch
    ce(m(torch.from_numpy(xb)), torch.from_numpy(yb)).backward()
    opt.step()

print("sau 200 bước:")
print("W1:", np.abs(net.W1 - m[0].weight.detach().numpy().T).max())
print("b1:", np.abs(net.b1 - m[0].bias.detach().numpy()).max())
print("W2:", np.abs(net.W2 - m[2].weight.detach().numpy().T).max())
print("b2:", np.abs(net.b2 - m[2].bias.detach().numpy()).max())

sau 200 bước:
W1: 1.1102230246251565e-16
b1: 3.469446951953614e-17
W2: 2.220446049250313e-16
b2: 6.938893903907228e-17


In [5]:
import time
net0 = MLP(n_hidden=128, seed=0)                         # float32, đúng cấu hình ngày 6
m = to_torch(net0, torch.float32)
opt = torch.optim.SGD(m.parameters(), lr=0.1)
ce = nn.CrossEntropyLoss()

Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr)
Xv, yv = torch.from_numpy(Xva), torch.from_numpy(yva)
rng = np.random.default_rng(0)                           # trùng seed của train_mlp
N = len(Xtr)

t0 = time.perf_counter()
for ep in range(20):
    perm = rng.permutation(N)                            # trùng thứ tự batch với train_mlp
    m.train()
    for s in range(0, N, 128):
        idx = torch.from_numpy(perm[s:s + 128])
        opt.zero_grad()
        ce(m(Xt[idx]), yt[idx]).backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        va = (m(Xv).argmax(1) == yv).float().mean().item()
    print(ep + 1, round(va * 100, 2))
print("thời gian PyTorch (có đánh giá mỗi epoch):", round(time.perf_counter() - t0, 1), "s")

1 90.59
2 93.1
3 93.98
4 94.5
5 95.36
6 95.71
7 95.88
8 96.19
9 96.4
10 96.57
11 96.68
12 96.71
13 96.93
14 96.94
15 96.96
16 97.09
17 97.12
18 97.18
19 97.18
20 97.18
thời gian PyTorch (có đánh giá mỗi epoch): 8.5 s


In [6]:
torch.manual_seed(0)
m2 = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.SGD(m2.parameters(), lr=0.1)
rng = np.random.default_rng(0)
for ep in range(20):
    perm = rng.permutation(N)
    for s in range(0, N, 128):
        idx = torch.from_numpy(perm[s:s + 128])
        opt.zero_grad(); ce(m2(Xt[idx]), yt[idx]).backward(); opt.step()
with torch.no_grad():
    print("val acc (init mặc định của PyTorch):", (m2(Xv).argmax(1) == yv).float().mean().item() * 100)

val acc (init mặc định của PyTorch): 97.08999991416931
